In [0]:
%sql
-- To ensure environment is ready
-- 1. Ensure the schema exists
CREATE SCHEMA IF NOT EXISTS workspace.dev_bronze_layer;

-- 2. Create the Volume specifically for your raw data
CREATE VOLUME IF NOT EXISTS workspace.dev_bronze_layer.raw_files;

CREATE SCHEMA IF NOT EXISTS workspace.dev_silver_layer;
CREATE SCHEMA IF NOT EXISTS workspace.dev_gold_layer;

-- 1. Drop the existing table
DROP TABLE IF EXISTS workspace.dev_bronze_layer.events_raw;


In [0]:
# Create the landing folder inside your new volume
dbutils.fs.mkdirs("/Volumes/workspace/dev_bronze_layer/raw_files/landing")

In [0]:
# Filename: src/notebooks/bronze_ingestion.py
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# 1. Define the landing path in Volume
landing_path = "/Volumes/workspace/dev_bronze_layer/raw_files/landing/"
checkpoint_path = "/Volumes/workspace/dev_bronze_layer/raw_files/_checkpoints/bronze/"

# Define the schema explicitly based on your e-commerce data
event_schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .schema(event_schema) # Add this to bypass the empty directory error
  .load(landing_path)
  .select("*", "_metadata.file_path")
  .withColumnRenamed("file_path", "source_file")
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("workspace.dev_bronze_layer.events_raw"))